# Quickstart: Training pinnrl on a `The Well` benchmark dataset

This notebook walks through training a neural surrogate on `active_matter` from [Polymathic AI's `the_well`](https://github.com/PolymathicAI/the_well). The Well dataset has no analytical solver in pinnrl, so we use the **`data_only`** mode — the network learns to reproduce the reference field directly, without a PDE residual term.

Prerequisites:

```bash
pip install 'pinnrl[well]'
```

If you have already downloaded the dataset locally with `the-well-download`, point `base=` at it. Otherwise the loader streams from `hf://datasets/polymathic-ai/`.

In [1]:
import sys, os

# Make pinnrl importable when running from notebooks/
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)


## 1. Inspect the registry

In [2]:
from pinnrl.datasets import WELL_REGISTRY, get_entry

for name, entry in sorted(WELL_REGISTRY.items()):
    print(f"{name:35s}  {entry.n_spatial_dims}D  mode={entry.recommended_mode:14s}  arch={entry.default_architecture}")

entry = get_entry('active_matter')
print('\nactive_matter fields:', entry.fields)
print('domain:', entry.domain, 'time:', entry.time_domain)

MHD_64                               3D  mode=data_only       arch=mlp
acoustic_scattering_maze             2D  mode=data_augmented  arch=fno
active_matter                        2D  mode=data_only       arch=fno
euler_multi_quadrants_periodicBC     2D  mode=data_only       arch=fno
gray_scott_reaction_diffusion        2D  mode=data_only       arch=fno
helmholtz_staircase                  2D  mode=data_augmented  arch=fno
planetswe                            2D  mode=data_only       arch=fno
rayleigh_benard                      2D  mode=data_only       arch=fno
rayleigh_taylor_instability          3D  mode=data_only       arch=mlp
shear_flow                           2D  mode=data_only       arch=fno
turbulent_radiative_layer_2D         2D  mode=data_only       arch=fno
viscoelastic_instability             2D  mode=data_only       arch=fno

active_matter fields: ('concentration', 'velocity_x', 'velocity_y', 'orientation_xx', 'orientation_xy', 'orientation_yx', 'orientation_yy', 'strain

## 2. Load a small slice into memory

`load_well_slice` flattens the trajectory grid into the same `(x, t, u)` triple pinnrl uses everywhere — so the rest of the framework treats it identically to synthetic observations.

In [3]:
from pinnrl.datasets import load_well_slice

slice_ = load_well_slice(
    name='active_matter',
    split='train',
    n_traj=2,
    n_points=5_000,
    seed=0,
    device='cpu',
)
for k, v in slice_.items():
    print(f'{k}: shape={tuple(v.shape)}  dtype={v.dtype}')

x: shape=(5000, 2)  dtype=torch.float32
t: shape=(5000, 1)  dtype=torch.float32
u: shape=(5000, 11)  dtype=torch.float32


## 3. Train an FNO in data_only mode

The fastest path is the CLI: `build_config_dict` overlays the registry defaults and the trainer takes it from there.

In [4]:
import yaml, torch
from pathlib import Path
from pinnrl.training.train import build_config_dict, run_training

yaml_cfg = yaml.safe_load(Path('../pinnrl/config/config.yaml').read_text())
yaml_cfg.setdefault('training', {})['num_epochs'] = 200

config_dict = build_config_dict(
    yaml_cfg,
    pde_name='Heat Equation',  # Placeholder — overridden by dataset defaults.
    arch_type='fno',
    use_rl=False,
    epochs=200,
    dataset={
        'name': 'active_matter',
        'split': 'train',
        'n_traj': 2,
        'n_points': 5_000,
        'seed': 0,
        'base': None,
        'use_defaults': True,
    },
)
print('mode:', config_dict['training']['mode'])
print('input_dim/output_dim:', config_dict['model']['input_dim'], '/', config_dict['model']['output_dim'])

device = torch.device('cpu')
config_dict['device'] = str(device)
run_training(config_dict, device)

mode: data_only
input_dim/output_dim: 3 / 11
Experiment: 20260501_231933_active_matter_fno_no_rl
Directory: experiments/20260501_231933_active_matter_fno_no_rl


2026-05-01 23:19:33,868 - INFO - Adaptive weights are disabled


2026-05-01 23:19:33,869 - INFO - ==================================================


2026-05-01 23:19:33,870 - INFO - STARTING TRAINING FOR PDE: HeatEquation


2026-05-01 23:19:33,870 - INFO - NEURAL NETWORK ARCHITECTURE: fno


...

2026-05-01 23:22:59,768 - WARNING - Error plotting training history: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



2026-05-01 23:22:59,768 - INFO - Training history plot saved successfully


2026-05-01 23:22:59,872 - ERROR - Error in plot_solution_comparison: shape '[100, 100]' is invalid for input of size 110000


2026-05-01 23:22:59,873 - ERROR - Error saving plots: shape '[100, 100]' is invalid for input of size 110000


Training completed successfully.


## 4. Plot prediction vs. reference

After training completes, the latest experiment directory under `results_dir` contains `final_model.pt` and a `live_snapshot.npz`. Reload them and compare a single field channel.

In [5]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

results_dir = Path(yaml_cfg['paths']['results_dir'])
experiment = sorted(results_dir.glob('*active_matter*'))[-1]
snapshot = np.load(experiment / 'live_snapshot.npz')
u_pred = snapshot['u_pred']
u_ref = snapshot.get('u_ref', np.zeros_like(u_pred))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(u_pred, cmap='viridis')
axes[0].set_title('Prediction (channel 0)')
axes[1].imshow(u_ref, cmap='viridis')
axes[1].set_title('Reference')
plt.tight_layout()
plt.show()

/var/folders/yx/b_pyg98x1dj97pbflpck32d80000gn/T/ipykernel_43187/3523696128.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Try a matched dataset (`acoustic_scattering_maze`)

For datasets whose default PDE matches an existing pinnrl solver, switch to `data_augmented` to combine the residual loss with the dataset's reference field. This is the regime where pinnrl's RL adaptive sampler can be benchmarked against the ground truth from The Well.

In [6]:
matched = build_config_dict(
    yaml_cfg,
    pde_name='Wave Equation',
    arch_type='fno',
    use_rl=True,
    epochs=300,
    dataset={
        'name': 'acoustic_scattering_maze',
        'split': 'train',
        'n_traj': 1,
        'n_points': 4_000,
        'seed': 0,
        'base': None,
        'use_defaults': True,
    },
)
print('mode:', matched['training']['mode'])  # data_augmented
print('observation_data source:', matched['pde']['observation_data']['source'])

mode: data_augmented
observation_data source: well
